# 01 · 数据：CASIA-WebFace（训练）+ LFW 等验证集（测试）

本 notebook 完成实验 **E0**：检查数据格式、建立缓存、划分留出身份、生成固定遮挡、测吞吐。

| 数据 | 来源（Kaggle） | 内容 |
|---|---|---|
| CASIA-WebFace 112×112 | `yakhyokhuja/webface-112x112`（2.7 GB zip） | `webface_112x112/id_<k>/<k>_<j>.jpg`：490,623 张、10,572 个身份，已按五点模板对齐 |
| 验证集 112×112 | `yakhyokhuja/agedb-30-calfw-cplfw-lfw-aligned-112x112`（1.4 GB zip） | LFW / AgeDB-30 / CALFW / CPLFW 各 6000 对（12000 张 bmp + `<name>_ann.txt`），第 t 对 = 图 2t 与 2t+1，按 10 折 × 600 对排列 |

**下载**（约 4.1 GB，无需 Kaggle 账号）放到 `data/_downloads/`：
```
curl -L -o data/_downloads/webface.zip https://www.kaggle.com/api/v1/datasets/download/yakhyokhuja/webface-112x112
curl -L -o data/_downloads/val.zip     https://www.kaggle.com/api/v1/datasets/download/yakhyokhuja/agedb-30-calfw-cplfw-lfw-aligned-112x112
```
zip 不需要解压：`prepare_data` 直接从 zip 解码，一次性写成 `data/cache/casia_112.npy`（uint8 memmap，18.5 GB，12 进程约 1.5 分钟）和 `pairs_<name>.npz`。

## 0. 环境检查

In [ ]:
%load_ext autoreload
%autoreload 2
# autoreload：修改 src/occprune 下的代码后无需重启 kernel

import os
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src" / "occprune").exists() else Path.cwd().parent  # 仓库根目录
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)   # 相对路径（data/、results/）都以仓库根目录为准

import occprune
print("Python  :", sys.executable)
print("occprune:", occprune.__file__)

## 1. 建缓存、划分、固定遮挡
- **留出身份**：从 ≥5 张图的身份中随机留出 2%（211 个身份、9,413 张）**不参与训练**，在其上构造 3000 正对 + 3000 负对，用于每个 epoch 的模型选择。LFW 只做最终测试，从不参与选择。
- **遮挡**：每对的**第二张图**加遮挡（遮挡的查询图 vs 干净的底库图）。随机方块 10/30/50% 用固定种子；按区域遮挡（双眼、口鼻、上半脸、下半脸）用 112×112 对齐模板的固定坐标。所有遮挡保存在 `results/masks/`。

In [ ]:
from occprune.utils import load_config
from occprune.pipeline import prepare_data, throughput_test

cfg = load_config()                 # configs/default.yaml
stats = prepare_data(cfg)
{k: v for k, v in stats.items() if k not in ("casia", "region_rects_y0x0hw")}

In [ ]:
stats["casia"]

In [ ]:
from IPython.display import Image
Image("figures/occlusion_conditions.png", width=900)

## 2. 吞吐测试
`loader` = DataLoader 从 memmap 读取（含 worker 启动时间）；`train_step` = GPU 上一步训练（bf16 autocast）。
实测（RTX 3070 Ti Laptop）：训练约 5000 张/秒；loader 稳态 > 2 万张/秒，但 Windows 上每个 epoch 启动 worker 需约 20 秒。一个 epoch 约 2 分钟。

In [ ]:
tp = throughput_test(cfg, steps=80)